# MediaCloud API Explorer

Reference: see `API_USAGE.md` in this folder.

In [ ]:
from urllib.parse import urlencode

# Human-readable source label used in notebook output.
SOURCE_NAME = "MediaCloud (v4 Search API metadata)"
# Active MediaCloud v4 story-list endpoint.
BASE_URL = "https://search.mediacloud.org/api/search/story-list"
# High-level capability notes for quick orientation before making live calls.
CAPABILITIES = [
    "Cross-platform story search via MediaCloud API v4.",
    "Boolean query syntax in q=... with start/end date filters.",
    "Story-level metadata (title, URL, publish date, media metadata).",
    "Pagination support via pagination_token in API response.",
]
# Default query parameters chosen to return a small, recent slice for sanity checks.
DEFAULT_QUERY_PARAMS = {
    "q": 'inflation OR "interest rates"',
    "start": "2025-01-01",
    "end": "2025-01-31",
    "platform": "onlinenews-mediacloud",
    "page_size": "10",
}

print(SOURCE_NAME)
print("Preview only: this cell does not fetch API records.")
print("Base URL:")
print("-", BASE_URL)
print("Capabilities:")
for item in CAPABILITIES:
    print("-", item)
print("\nDefault query params:")
for key, value in DEFAULT_QUERY_PARAMS.items():
    print(f"- {key}: {value}")

print("\nAuthentication expectation:")
print("- Use HTTP header Authorization: Token <MEDIACLOUD_API_KEY>.")
print("- API key is never added to the URL query string.")

print("\nRequest URL preview (no key in URL):")
print(f"{BASE_URL}?{urlencode(DEFAULT_QUERY_PARAMS, doseq=True)}")

In [ ]:
from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlparse
from urllib.request import Request, urlopen
import json
import os
import re
import socket
import time


def normalize_env_value(raw_value: str) -> str:
    """Trim matching quote wrappers from .env values like 'value' or "value"."""
    if (
        len(raw_value) >= 2
        and raw_value[0] == raw_value[-1]
        and raw_value[0] in {'"', "'"}
    ):
        return raw_value[1:-1]
    return raw_value


def find_project_root(start_dir: Path) -> Path | None:
    """Find the repository root by walking upward until a .env file is found."""
    for directory in [start_dir, *start_dir.parents]:
        if (directory / ".env").exists():
            return directory
    return None


def load_root_env() -> Path | None:
    """Load key/value pairs from root .env into os.environ without overriding existing vars."""
    project_root = find_project_root(Path.cwd())
    if project_root is None:
        return None
    env_path = project_root / ".env"
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), normalize_env_value(value.strip()))
    return project_root


def detect_text_fields(stories: list[dict]) -> dict[str, int]:
    """Count how many returned stories contain potential text-bearing fields."""
    candidates = ["text", "content", "body", "description", "summary"]
    counts: dict[str, int] = {field: 0 for field in candidates}
    for story in stories:
        for field in candidates:
            value = story.get(field)
            if isinstance(value, str) and value.strip():
                counts[field] += 1
    return counts


def endpoint_host(endpoint_url: str) -> str:
    """Return the hostname for an endpoint URL."""
    return urlparse(endpoint_url).hostname or ""


def dns_check(endpoint_url: str) -> None:
    """Print DNS check diagnostics for the endpoint host."""
    host = endpoint_host(endpoint_url)
    if not host:
        print(f"DNS check skipped: no host in URL {endpoint_url}")
        return
    try:
        socket.getaddrinfo(host, 443)
        print(f"DNS check: {host} resolved successfully.")
    except socket.gaierror as dns_error:
        print(f"DNS check failed for {host}: {dns_error}")


def date_from_iso8601(raw_value: str) -> str:
    """Convert an ISO-8601 datetime/date string into YYYY-MM-DD date format."""
    date_part = raw_value.strip()[:10]
    datetime.strptime(date_part, "%Y-%m-%d")
    return date_part


def convert_legacy_query_params(raw_params: dict) -> dict:
    """Convert legacy v2-style params (rows/fq/wc) into v4 story-list params."""
    params = dict(raw_params)

    # Map legacy `rows` to v4 `page_size` when page_size is not explicitly set.
    if "rows" in params and "page_size" not in params:
        params["page_size"] = str(params["rows"])

    # Parse first publish_date filter when params are still in legacy `fq` format.
    if "fq" in params and ("start" not in params or "end" not in params):
        fq_values = params["fq"] if isinstance(params["fq"], list) else [params["fq"]]
        pattern = re.compile(r"^publish_date:\[(.+?) TO (.+?)\]$")
        for fq in fq_values:
            match = pattern.match(str(fq).strip())
            if match:
                params["start"] = date_from_iso8601(match.group(1))
                params["end"] = date_from_iso8601(match.group(2))
                break

    # Drop v2-only keys that the v4 endpoint does not accept.
    for key in ["rows", "fq", "wc", "sort"]:
        params.pop(key, None)

    # Ensure required v4 query keys are present.
    params.setdefault("start", "2025-01-01")
    params.setdefault("end", "2025-01-31")
    params.setdefault("platform", "onlinenews-mediacloud")
    params.setdefault("page_size", "10")

    return params


project_root = load_root_env()

# Pull defaults from earlier cells when available so users can override interactively.
base_url = normalize_env_value(
    os.getenv(
        "MEDIACLOUD_BASE_URL",
        globals().get(
            "BASE_URL", "https://search.mediacloud.org/api/search/story-list"
        ),
    ).strip()
)
default_query_params = globals().get(
    "DEFAULT_QUERY_PARAMS",
    {
        "q": 'inflation OR "interest rates"',
        "start": "2025-01-01",
        "end": "2025-01-31",
        "platform": "onlinenews-mediacloud",
        "page_size": "10",
    },
)

query_params = convert_legacy_query_params(default_query_params)

# MediaCloud v4 uses Authorization: Token <key> header authentication.
api_key = normalize_env_value(os.getenv("MEDIACLOUD_API_KEY", "").strip())
if not api_key:
    print("Missing MEDIACLOUD_API_KEY; set it to fetch live records.")
else:
    dns_check(base_url)

    request_url = f"{base_url}?{urlencode(query_params, doseq=True)}"

    # Safe URL log excludes credentials because auth is header-based in v4.
    print("Live request URL (no key in URL):")
    print(request_url)

    payload = None
    last_error = None
    # Retry strategy: immediate attempt, then short backoff windows for transient limits.
    for attempt_index, delay_seconds in enumerate([0, 4, 8], start=1):
        if delay_seconds > 0:
            print(f"Waiting {delay_seconds}s before retry...")
            time.sleep(delay_seconds)
        try:
            request = Request(
                request_url,
                headers={
                    "User-Agent": "news-api-explorer/1.0",
                    "Accept": "application/json",
                    "Authorization": f"Token {api_key}",
                },
            )
            with urlopen(request, timeout=30) as response:
                payload = json.loads(response.read().decode("utf-8"))
            print(f"Fetch succeeded on attempt {attempt_index}.")
            break
        except HTTPError as error:
            last_error = error
            print(f"Attempt {attempt_index} failed with HTTP {error.code}.")
            try:
                body = error.read().decode("utf-8", errors="replace").strip()
            except Exception:
                body = ""
            if body:
                print(f"Response body: {body}")
            # Retry throttling (429) and temporary server failures (5xx).
            if error.code != 429 and not (500 <= error.code <= 599):
                break
        except URLError as error:
            last_error = error
            print(f"Attempt {attempt_index} failed: {error}")
            break
        except Exception as error:
            last_error = error
            print(f"Attempt {attempt_index} failed: {error}")
            break

    if payload is None:
        print("No live payload returned.")
        print(f"Last error: {last_error}")
    else:
        # search/story-list returns {stories: [...], pagination_token: ...}. Keep safe fallbacks.
        if isinstance(payload, list):
            stories = payload
            pagination_token = None
        elif isinstance(payload, dict):
            stories = (
                payload.get("stories")
                or payload.get("results")
                or payload.get("items")
                or []
            )
            pagination_token = payload.get("pagination_token")
        else:
            stories = []
            pagination_token = None

        print(f"\nStories returned: {len(stories)}")
        if pagination_token:
            print(f"pagination_token present: {pagination_token[:32]}...")
        for index, story in enumerate(stories[:5], start=1):
            title = story.get("title") or story.get("name") or "<missing title>"
            url = story.get("url") or story.get("link") or "<missing url>"
            publish_date = story.get("publish_date") or "<missing publish_date>"
            print(f"{index}. {title}")
            print(f"   URL: {url}")
            print(f"   publish_date: {publish_date}")

        if stories:
            text_field_counts = detect_text_fields(stories)
            print("\nPotential direct-text fields present in this response batch:")
            for field_name, field_count in text_field_counts.items():
                print(f"- {field_name}: {field_count}/{len(stories)} stories")
            if all(count == 0 for count in text_field_counts.values()):
                print(
                    "No direct article body text fields detected in returned story objects."
                )

        # Persist raw payload for schema inspection and downstream parser design.
        if project_root is None:
            output_dir = Path("outputs")
        else:
            output_dir = (
                project_root / "notebooks/api_explorer" / "mediacloud" / "outputs"
            )
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / "mediacloud_live_response.json"
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        print(f"\nSaved payload to: {output_path.resolve()}")